# Exploratory Data Analysis: Generative AI Adoption and Perception in Software Engineering

This notebook delivers a **deep, reproducible EDA** of survey datasets stored under `base_files/<cohort>/responses<stage>.csv`.

## Objective
Understand **adoption** and **perception** of generative AI in software engineering, with emphasis on:

- respondent profile and experience context
- overall and software-engineering-specific AI usage
- tools and activities associated with generative AI adoption
- perceived usefulness, productivity, quality, process change, and skill implications
- perceived impact on software engineering roles
- exploratory subgroup comparisons and light statistical testing

## Important notes
- Each loaded response receives two provenance columns: `cohort` and `stage`.
- The loader reads only files matching `responses<stage>.csv` directly under each cohort folder in `base_files`.
- Personally identifying columns are removed from the analytical dataset.
- Two duplicated fields with suffix `.1` are treated as import artifacts. The original versions are fully empty; the `.1` versions contain the actual responses.
- Open-ended text responses are **kept in the cleaned dataset** for completeness but **excluded from qualitative text analysis**, per project requirements.
- Statistical tests in this notebook are **exploratory**, not confirmatory, due to the small sample size.


In [ ]:
# Core imports
from pathlib import Path
import re
import textwrap
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from scipy.stats import (
    spearmanr,
    mannwhitneyu,
    kruskal,
    chi2_contingency
)

# Reproducibility / display config
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

BASE_FILES_DIR = Path("base_files")
OUTPUT_DIR = Path("eda_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

STAGE_FILE_PATTERN = re.compile(r"^responses(\d+)\.csv$", re.IGNORECASE)


In [ ]:
def extract_stage_from_filename(file_path):
    match = STAGE_FILE_PATTERN.match(file_path.name)
    if match:
        return int(match.group(1))
    return pd.NA


def load_survey_data(base_files_dir=BASE_FILES_DIR):
    if not base_files_dir.exists():
        raise FileNotFoundError(
            "The base_files directory was not found. Expected cohort data in "
            "'base_files/<cohort>/responses<stage>.csv'."
        )

    source_records = []
    frames = []

    cohort_dirs = sorted(
        [path for path in base_files_dir.iterdir() if path.is_dir()],
        key=lambda path: path.name
    )

    for cohort_dir in cohort_dirs:
        cohort = cohort_dir.name
        csv_files = sorted(
            [
                path for path in cohort_dir.iterdir()
                if path.is_file() and STAGE_FILE_PATTERN.match(path.name)
            ],
            key=lambda path: (extract_stage_from_filename(path), path.name)
        )

        for csv_path in csv_files:
            stage = extract_stage_from_filename(csv_path)
            frame = pd.read_csv(csv_path)
            frame["cohort"] = cohort
            frame["stage"] = stage
            frames.append(frame)

            source_records.append({
                "cohort": cohort,
                "stage": stage,
                "file_name": csv_path.name,
                "relative_path": str(csv_path.as_posix()),
                "n_rows": len(frame),
                "n_columns": frame.shape[1]
            })

    if not frames:
        raise FileNotFoundError(
            "No files matching 'responses<stage>.csv' were found under base_files/<cohort>."
        )

    raw = pd.concat(frames, ignore_index=True, sort=False)
    source_files = pd.DataFrame(source_records).sort_values(
        ["cohort", "stage", "file_name"]
    ).reset_index(drop=True)

    return raw, source_files


# Load the raw dataset(s)
raw, source_files = load_survey_data()

print(f"Loaded {len(source_files)} source file(s)")
display(source_files)

print(f"Raw shape: {raw.shape[0]} rows x {raw.shape[1]} columns")
display(raw.head(3))


## 1. Initial schema audit

This section checks:
- dataset size
- column names
- duplicated fields introduced by import/export
- obvious missingness issues


In [ ]:
schema = pd.DataFrame({
    "column": raw.columns,
    "dtype": raw.dtypes.astype(str).values,
    "non_null": raw.notna().sum().values,
    "missing": raw.isna().sum().values,
    "missing_pct": (raw.isna().mean().values * 100).round(2)
})
display(schema)


In [ ]:
# Identify import-style duplicated columns, such as 'question' and 'question.1'
duplicate_like_cols = [c for c in raw.columns if c.endswith(".1")]
duplicate_audit = []

for dup_col in duplicate_like_cols:
    base_col = dup_col[:-2]
    base_exists = base_col in raw.columns
    duplicate_audit.append({
        "base_column": base_col,
        "duplicate_column": dup_col,
        "base_exists": base_exists,
        "base_non_null": int(raw[base_col].notna().sum()) if base_exists else np.nan,
        "duplicate_non_null": int(raw[dup_col].notna().sum()),
        "base_all_missing": bool(raw[base_col].isna().all()) if base_exists else np.nan
    })

duplicate_audit = pd.DataFrame(duplicate_audit)
display(duplicate_audit)

if not duplicate_audit.empty:
    print("Interpretation:")
    for _, row in duplicate_audit.iterrows():
        print(
            f"- Base column all missing? {row['base_all_missing']} | "
            f"Using populated duplicate: {row['duplicate_column']}"
        )
